<a href="https://colab.research.google.com/github/d005810/ECAA08-Manufatura-Flexivel/blob/main/etapa-03-arvores/Aula%2027%20-%20Arvores%20de%20Decisao%20-%20Algoritmos%20Classicos%20de%20Construcao%20de%20Arvore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 27: Arvores de Decisao -- Algoritmos Classicos de Construcao de Arvore
## Inducao de Regras (ID3, C4.5 e CART), Poda e Supressao Hierarquica de Alarmes na Celula de Manufatura Flexivel (FMS)

---

### 1. Contexto na Engenharia de Controle e Automacao
Na arquitetura do projeto **SCADA-Core Manufatura Flexivel (`ECAA08-MANUFATURA-FLEXIVEL`)**, a planta opera em tres setores integrados:
* **Setor 100 (Alimentacao):** Silo (`SILO-100`, `LS-101`, `ZS-101`, pistao `XV-101`) e motor da esteira principal (`M-101`, rele termico `TH-101`);
* **Setor 200 (Inspecao e Triagem):** Tunel optico (`INSP-200`, formato `ZS-201/202`, cor RGB `AS-201..203`) e desviadores pneumaticos (`DIV-201..203`, `XV-201..203`);
* **Setor 300 (Loteamento e Seguranca):** Caixas de loteamento (`CX-301..303`, `US-301..303`, limite de 10 pecas), calha de descarte (`DESC-304`), emergencia (`HS-301` / `ESD-301`) e sinaleiro de alarme geral (`HS-302`).

Neste notebook, implementamos **100% do zero (sem bibliotecas externas de Machine Learning)** os tres algoritmos classicos de construcao top-down de arvores de decisao (**ID3**, **C4.5** e **CART**) para resolver dois problemas industriais reais:
1. **Estudo de Caso 1 (Discreto e Auditavel):** Inducao da arvore de roteamento e intertravamento da estacao `INSP-200`, demonstrando na pratica como o **vies de alta cardinalidade do ID3** e corrigido pela **Razao de Ganho (C4.5)** e pelas **biparticoes de Gini (CART)**;
2. **Estudo de Caso 2 (Telemetria Continua, Poda e Mitigacao de Inundacao de Alarmes):** Deteccao de causa raiz a partir de sensores analogicos (`Pressao_Ar_bar`, `Corrente_M101_A`, `Tempo_Inspecao_ms`) e discretos (`ESD_301`, `LS_101`), demonstrando busca de limiar otimo, **pos-poda (*Reduced-Error Pruning*)** contra ruido industrial e traducao automatica da arvore para **Texto Estruturado IEC 61131-3**.

In [ ]:
# Importacao de Bibliotecas Padrao e Cientificas
import math
from collections import Counter
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Set, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# Configuracao visual padrao dos graficos SCADA-Core
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Ambiente Python inicializado para o modulo de Arvores de Decisao da FMS.")

---
### 2. Fundamentacao Matematica das Metricas de Pureza e Particao

Dado um conjunto de registros $D$ com $C$ classes e frequencias relativas $p_k = \frac{|D_k|}{|D|}$:

1. **Entropia de Shannon (ID3 / C4.5):**
   $$H(D) = -\sum_{k=1}^{C} p_k \log_2(p_k)$$
2. **Ganho de Informacao (ID3):**
   $$IG(D, A) = H(D) - \sum_{v \in \text{Valores}(A)} \frac{|D_v|}{|D|} H(D_v)$$
3. **Informacao de Divisao e Razao de Ganho (C4.5):**
   $$SplitInfo(D, A) = -\sum_{v \in \text{Valores}(A)} \frac{|D_v|}{|D|} \log_2\left(\frac{|D_v|}{|D|}\right), \qquad GR(D, A) = \frac{IG(D, A)}{SplitInfo(D, A)}$$
4. **Impureza de Gini e Reducao em Biparticao Binaria (CART):**
   $$Gini(D) = 1 - \sum_{k=1}^{C} p_k^2, \qquad \Delta Gini(D, A) = Gini(D) - \left(\frac{|D_E|}{|D|}Gini(D_E) + \frac{|D_D|}{|D|}Gini(D_D)\right)$$

In [ ]:
def shannon_entropy(labels: List[Any]) -> float:
    """Calcula a Entropia de Shannon H(D) em bits."""
    n = len(labels)
    if n == 0:
        return 0.0
    counts = Counter(labels)
    entropy = 0.0
    for count in counts.values():
        if count > 0:
            p = count / n
            entropy -= p * math.log2(p)
    return entropy


def gini_impurity(labels: List[Any]) -> float:
    """Calcula a Impureza de Gini(D) = 1 - sum(p_k^2)."""
    n = len(labels)
    if n == 0:
        return 0.0
    counts = Counter(labels)
    sum_sq = sum((count / n) ** 2 for count in counts.values())
    return 1.0 - sum_sq


def evaluate_categorical_multi_split(values: List[Any], labels: List[Any]) -> Dict[str, float]:
    """
    Avalia uma particao multi-ramo sobre atributo categorico:
    retorna H_cond, IG (ID3), SplitInfo e Gain Ratio (C4.5).
    """
    n = len(labels)
    parent_h = shannon_entropy(labels)
    groups: Dict[Any, List[Any]] = {}
    for val, lbl in zip(values, labels):
        groups.setdefault(val, []).append(lbl)

    h_cond = 0.0
    split_info = 0.0
    for grp_labels in groups.values():
        w = len(grp_labels) / n
        h_cond += w * shannon_entropy(grp_labels)
        if w > 0:
            split_info -= w * math.log2(w)

    ig = parent_h - h_cond
    gr = (ig / split_info) if split_info > 1e-12 else 0.0
    return {
        "H_cond": h_cond,
        "IG": ig,
        "SplitInfo": split_info,
        "GainRatio": gr,
    }


def best_binary_categorical_split(
    values: List[Any], labels: List[Any], min_samples_leaf: int = 1
) -> Tuple[Optional[Any], float, float]:
    """
    Para CART em atributo categorico, avalia todas as biparticoes (val == c vs val != c)
    e retorna (melhor_categoria, delta_gini, gini_split).
    """
    n = len(labels)
    parent_gini = gini_impurity(labels)
    unique_vals = sorted(set(values), key=str)

    best_cat = None
    best_gain = -1.0
    best_gini_split = parent_gini

    for cat in unique_vals:
        left_labels = [lbl for val, lbl in zip(values, labels) if val == cat]
        right_labels = [lbl for val, lbl in zip(values, labels) if val != cat]
        if len(left_labels) < min_samples_leaf or len(right_labels) < min_samples_leaf:
            continue

        gini_split = (
            (len(left_labels) / n) * gini_impurity(left_labels)
            + (len(right_labels) / n) * gini_impurity(right_labels)
        )
        delta_gini = parent_gini - gini_split
        if delta_gini > best_gain + 1e-12:
            best_gain = delta_gini
            best_cat = cat
            best_gini_split = gini_split

    return best_cat, best_gain, best_gini_split


def best_continuous_threshold(
    values: List[float],
    labels: List[Any],
    algorithm: str = "cart",
    min_samples_leaf: int = 1,
) -> Tuple[Optional[float], float, float]:
    """
    Encontra o limiar numerico otimo t (<= t vs > t) para C4.5 ou CART.
    Retorna (melhor_limiar, metrica_selecao, ganho_informacao_ou_delta_gini).
    Nota (Quinlan, 1993): no C4.5, escolhe-se t* maximizando IG e compara-se atributos via GR(t*).
    """
    n = len(labels)
    if n < 2 * min_samples_leaf:
        return None, -1.0, 0.0

    paired = sorted(zip(values, labels), key=lambda x: x[0])
    sorted_vals = [p[0] for p in paired]
    sorted_lbls = [p[1] for p in paired]

    parent_h = shannon_entropy(sorted_lbls)
    parent_gini = gini_impurity(sorted_lbls)

    best_t = None
    best_score = -1.0
    best_aux_gr = 0.0

    for i in range(n - 1):
        # So ha limiar valido entre valores numericamente distintos
        if abs(sorted_vals[i + 1] - sorted_vals[i]) < 1e-12:
            continue
        left_count = i + 1
        right_count = n - left_count
        if left_count < min_samples_leaf or right_count < min_samples_leaf:
            continue

        threshold = 0.5 * (sorted_vals[i] + sorted_vals[i + 1])
        left_lbls = sorted_lbls[:left_count]
        right_lbls = sorted_lbls[left_count:]

        w_l = left_count / n
        w_r = right_count / n

        if algorithm == "cart":
            gini_split = w_l * gini_impurity(left_lbls) + w_r * gini_impurity(right_lbls)
            delta_gini = parent_gini - gini_split
            if delta_gini > best_score + 1e-12:
                best_score = delta_gini
                best_t = threshold
                best_aux_gr = delta_gini
        else:
            # C4.5: seleciona limiar pelo maior IG e calcula Gain Ratio correspondente
            h_cond = w_l * shannon_entropy(left_lbls) + w_r * shannon_entropy(right_lbls)
            ig = parent_h - h_cond
            split_info = -(w_l * math.log2(w_l) + w_r * math.log2(w_r))
            gr = (ig / split_info) if split_info > 1e-12 else 0.0
            if ig > best_score + 1e-12:
                best_score = ig
                best_t = threshold
                best_aux_gr = gr

    if best_t is None:
        return None, -1.0, 0.0
    selection_metric = best_aux_gr if algorithm == "c45" else best_score
    return best_t, selection_metric, best_score


# Testes rapidos de sanidade matematica
assert abs(shannon_entropy(["A", "A", "B", "B"]) - 1.0) < 1e-9
assert abs(gini_impurity(["A", "A", "B", "B"]) - 0.5) < 1e-9
assert abs(shannon_entropy(["A", "A", "A"]) - 0.0) < 1e-9
print("Funcoes matematicas de Entropia, IG, Gain Ratio e Gini validadas com sucesso.")

---
### 3. Implementacao Orientada a Objetos dos Algoritmos ID3, C4.5 e CART

A classe `ClassicDecisionTreeClassifier` unifica a inducao recursiva TDIDT para os tres algoritmos classicos:
* `algorithm='id3'`: atributos categoricos com divisao multi-ramo e selecao por maior **Ganho de Informacao ($IG$)**;
* `algorithm='c45'`: atributos categoricos (multi-ramo) e continuos (limiar binario $t^*$), selecao por **Razao de Ganho ($GR$)** e suporte a **pos-poda (*Reduced-Error Pruning*)**;
* `algorithm='cart'`: divisao **estritamente binaria** em atributos categoricos (`== c` vs `!= c`) e continuos (`<= t` vs `> t`), selecao por **Reducao de Gini ($\Delta Gini$)** e suporte a **pre-poda** e **pos-poda**.

In [ ]:
@dataclass
class DecisionNode:
    """Representa um no interno de decisao ou folha terminal na arvore."""
    is_leaf: bool
    predicted_class: Any
    samples: int
    class_counts: Dict[Any, int]
    impurity: float
    split_feature: Optional[str] = None
    split_type: Optional[str] = None  # 'multi', 'bin_cat', 'bin_num'
    split_threshold: Optional[float] = None
    split_category: Optional[Any] = None
    gain: float = 0.0
    children: Dict[Any, "DecisionNode"] = field(default_factory=dict)


class ClassicDecisionTreeClassifier:
    """
    Implementacao pura em Python dos algoritmos classicos ID3, C4.5 e CART
    para aplicacao em sistemas supervisorios SCADA e controle industrial.
    """

    def __init__(
        self,
        algorithm: str = "cart",
        max_depth: Optional[int] = None,
        min_samples_split: int = 2,
        min_samples_leaf: int = 1,
        min_impurity_decrease: float = 1e-7,
    ):
        algo = algorithm.lower()
        if algo not in {"id3", "c45", "cart"}:
            raise ValueError("Algoritmo deve ser 'id3', 'c45' ou 'cart'.")
        self.algorithm = algo
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.min_impurity_decrease = min_impurity_decrease
        self.root: Optional[DecisionNode] = None
        self.feature_names: List[str] = []
        self.categorical_features: Set[str] = set()

    def fit(
        self,
        X: pd.DataFrame,
        y: pd.Series,
        categorical_features: Optional[Set[str]] = None,
    ) -> "ClassicDecisionTreeClassifier":
        """Treina a arvore de decisao a partir de um DataFrame de atributos e serie alvo."""
        self.feature_names = list(X.columns)
        if categorical_features is not None:
            self.categorical_features = set(categorical_features)
        else:
            self.categorical_features = {
                col for col in X.columns if not pd.api.types.is_numeric_dtype(X[col])
            }
        self.root = self._build_tree(X.reset_index(drop=True), y.reset_index(drop=True), depth=0, used_multi=set())
        return self

    def _compute_impurity(self, labels: List[Any]) -> float:
        return gini_impurity(labels) if self.algorithm == "cart" else shannon_entropy(labels)

    def _majority_class(self, labels: List[Any]) -> Any:
        counts = Counter(labels)
        return sorted(counts.items(), key=lambda kv: (-kv[1], str(kv[0])))[0][0]

    def _build_tree(
        self,
        X: pd.DataFrame,
        y: pd.Series,
        depth: int,
        used_multi: Set[str],
    ) -> DecisionNode:
        labels = y.tolist()
        n = len(labels)
        counts = dict(Counter(labels))
        impurity = self._compute_impurity(labels)
        majority = self._majority_class(labels)

        # Criterios de parada: no puro, profundidade maxima ou amostras insuficientes
        if (
            len(counts) == 1
            or n < self.min_samples_split
            or (self.max_depth is not None and depth >= self.max_depth)
        ):
            return DecisionNode(
                is_leaf=True,
                predicted_class=majority,
                samples=n,
                class_counts=counts,
                impurity=impurity,
            )

        best_feature = None
        best_split_type = None
        best_gain = -1.0
        best_threshold = None
        best_category = None

        for feat in self.feature_names:
            vals = X[feat].tolist()
            if len(set(vals)) <= 1:
                continue

            if feat in self.categorical_features:
                if self.algorithm in {"id3", "c45"}:
                    if feat in used_multi:
                        continue
                    metrics = evaluate_categorical_multi_split(vals, labels)
                    grp_sizes = Counter(vals).values()
                    if min(grp_sizes) < self.min_samples_leaf:
                        continue
                    score = metrics["IG"] if self.algorithm == "id3" else metrics["GainRatio"]
                    if score > best_gain + 1e-12:
                        best_gain = score
                        best_feature = feat
                        best_split_type = "multi"
                        best_threshold = None
                        best_category = None
                else:
                    cat, d_gini, _ = best_binary_categorical_split(
                        vals, labels, min_samples_leaf=self.min_samples_leaf
                    )
                    if cat is not None and d_gini > best_gain + 1e-12:
                        best_gain = d_gini
                        best_feature = feat
                        best_split_type = "bin_cat"
                        best_threshold = None
                        best_category = cat
            else:
                if self.algorithm == "id3":
                    raise ValueError("ID3 classico nao suporta atributos continuos sem discretizacao previa.")
                t_opt, metric_score, _ = best_continuous_threshold(
                    vals, labels, algorithm=self.algorithm, min_samples_leaf=self.min_samples_leaf
                )
                if t_opt is not None and metric_score > best_gain + 1e-12:
                    best_gain = metric_score
                    best_feature = feat
                    best_split_type = "bin_num"
                    best_threshold = t_opt
                    best_category = None

        if best_feature is None or best_gain < self.min_impurity_decrease:
            return DecisionNode(
                is_leaf=True,
                predicted_class=majority,
                samples=n,
                class_counts=counts,
                impurity=impurity,
            )

        node = DecisionNode(
            is_leaf=False,
            predicted_class=majority,
            samples=n,
            class_counts=counts,
            impurity=impurity,
            split_feature=best_feature,
            split_type=best_split_type,
            split_threshold=best_threshold,
            split_category=best_category,
            gain=best_gain,
        )

        if best_split_type == "multi":
            next_used = set(used_multi)
            next_used.add(best_feature)
            for val in sorted(X[best_feature].unique(), key=str):
                mask = X[best_feature] == val
                node.children[val] = self._build_tree(
                    X[mask].reset_index(drop=True),
                    y[mask].reset_index(drop=True),
                    depth + 1,
                    next_used,
                )
        elif best_split_type == "bin_cat":
            mask_left = X[best_feature] == best_category
            node.children[True] = self._build_tree(
                X[mask_left].reset_index(drop=True),
                y[mask_left].reset_index(drop=True),
                depth + 1,
                used_multi,
            )
            node.children[False] = self._build_tree(
                X[~mask_left].reset_index(drop=True),
                y[~mask_left].reset_index(drop=True),
                depth + 1,
                used_multi,
            )
        elif best_split_type == "bin_num":
            mask_left = X[best_feature] <= best_threshold
            node.children[True] = self._build_tree(
                X[mask_left].reset_index(drop=True),
                y[mask_left].reset_index(drop=True),
                depth + 1,
                used_multi,
            )
            node.children[False] = self._build_tree(
                X[~mask_left].reset_index(drop=True),
                y[~mask_left].reset_index(drop=True),
                depth + 1,
                used_multi,
            )

        return node

    def _predict_node(self, node: DecisionNode, row: Dict[str, Any]) -> Tuple[Any, int]:
        """Retorna (classe_predita, numero_de_testes_executados)."""
        tests = 0
        curr = node
        while not curr.is_leaf:
            tests += 1
            val = row[curr.split_feature]
            if curr.split_type == "multi":
                if val in curr.children:
                    curr = curr.children[val]
                else:
                    return curr.predicted_class, tests
            elif curr.split_type == "bin_cat":
                curr = curr.children[val == curr.split_category]
            elif curr.split_type == "bin_num":
                curr = curr.children[val <= curr.split_threshold]
        return curr.predicted_class, tests

    def predict(self, X: pd.DataFrame) -> List[Any]:
        if self.root is None:
            raise RuntimeError("Modelo nao treinado.")
        return [self._predict_node(self.root, row)[0] for row in X.to_dict(orient="records")]

    def average_tests_per_sample(self, X: pd.DataFrame) -> float:
        if self.root is None or len(X) == 0:
            return 0.0
        total_tests = sum(self._predict_node(self.root, row)[1] for row in X.to_dict(orient="records"))
        return total_tests / len(X)

    def score(self, X: pd.DataFrame, y: pd.Series) -> float:
        preds = self.predict(X)
        return sum(p == yt for p, yt in zip(preds, y.tolist())) / len(y)

    def prune_reduced_error(self, X_val: pd.DataFrame, y_val: pd.Series) -> int:
        """
        Pos-poda estilo C4.5 (Reduced-Error Pruning) usando conjunto de validacao.
        Retorna a quantidade de subarvores colapsadas em folhas.
        """
        if self.root is None:
            return 0
        return self._prune_recursive(self.root, X_val.reset_index(drop=True), y_val.reset_index(drop=True))

    def _prune_recursive(self, node: DecisionNode, X_sub: pd.DataFrame, y_sub: pd.Series) -> int:
        if node.is_leaf or len(y_sub) == 0:
            return 0

        pruned_count = 0
        if node.split_type == "multi":
            for val, child in node.children.items():
                mask = X_sub[node.split_feature] == val
                pruned_count += self._prune_recursive(
                    child, X_sub[mask].reset_index(drop=True), y_sub[mask].reset_index(drop=True)
                )
        elif node.split_type == "bin_cat":
            mask = X_sub[node.split_feature] == node.split_category
            pruned_count += self._prune_recursive(
                node.children[True], X_sub[mask].reset_index(drop=True), y_sub[mask].reset_index(drop=True)
            )
            pruned_count += self._prune_recursive(
                node.children[False], X_sub[~mask].reset_index(drop=True), y_sub[~mask].reset_index(drop=True)
            )
        elif node.split_type == "bin_num":
            mask = X_sub[node.split_feature] <= node.split_threshold
            pruned_count += self._prune_recursive(
                node.children[True], X_sub[mask].reset_index(drop=True), y_sub[mask].reset_index(drop=True)
            )
            pruned_count += self._prune_recursive(
                node.children[False], X_sub[~mask].reset_index(drop=True), y_sub[~mask].reset_index(drop=True)
            )

        val_records = X_sub.to_dict(orient="records")
        y_list = y_sub.tolist()
        subtree_errors = sum(
            1 for rec, yt in zip(val_records, y_list) if self._predict_node(node, rec)[0] != yt
        )
        leaf_errors = sum(1 for yt in y_list if yt != node.predicted_class)

        if leaf_errors <= subtree_errors:
            node.is_leaf = True
            node.children = {}
            node.split_feature = None
            node.split_type = None
            node.split_threshold = None
            node.split_category = None
            pruned_count += 1

        return pruned_count

    def get_depth(self) -> int:
        def _d(n: Optional[DecisionNode]) -> int:
            if n is None or n.is_leaf:
                return 0
            return 1 + max(_d(c) for c in n.children.values())
        return _d(self.root)

    def count_nodes(self) -> int:
        def _c(n: Optional[DecisionNode]) -> int:
            if n is None:
                return 0
            return 1 + sum(_c(c) for c in n.children.values())
        return _c(self.root)

    def count_leaves(self) -> int:
        def _l(n: Optional[DecisionNode]) -> int:
            if n is None:
                return 0
            if n.is_leaf:
                return 1
            return sum(_l(c) for c in n.children.values())
        return _l(self.root)

    def print_tree(self) -> None:
        metric_name = "dGini" if self.algorithm == "cart" else ("GR" if self.algorithm == "c45" else "IG")
        def _p(n: DecisionNode, prefix: str = "", branch_lbl: str = "Raiz") -> None:
            if n.is_leaf:
                print(f"{prefix}[{branch_lbl}] -> FOLHA: {n.predicted_class} (n={n.samples}, imp={n.impurity:.4f})")
                return
            if n.split_type == "multi":
                cond = f"{n.split_feature}?"
            elif n.split_type == "bin_cat":
                cond = f"{n.split_feature} == '{n.split_category}'?"
            else:
                cond = f"{n.split_feature} <= {n.split_threshold:.3f}?"
            print(f"{prefix}[{branch_lbl}] ({cond} | {metric_name}={n.gain:.4f}, n={n.samples}, imp={n.impurity:.4f})")
            items = list(n.children.items())
            for k, child in items:
                lbl = "Sim" if (n.split_type != "multi" and k is True) else ("Nao" if (n.split_type != "multi" and k is False) else str(k))
                _p(child, prefix + "    ", lbl)
        if self.root is not None:
            _p(self.root)


print("Classe ClassicDecisionTreeClassifier (ID3, C4.5 e CART) pronta.")

---
### 4. Estudo de Caso 1: Roteamento e Intertravamento da Estacao `INSP-200` (Dataset Discreto)

Considere o conjunto de $N = 16$ ciclos historicos da FMS mapeados diretamente para as tags da Parte 1 e Parte 2:
* `Seg`: Integridade de seguranca (`OK` ou `Falha` via `HS-301`, `TH-101`, $\overline{s_a} \cdot s_b$ ou ambiguidade RGB);
* `Caixa`: Disponibilidade de loteamento (`Disp` $< 10$ pecas ou `Cheia` $= 10$ pecas via `US-301..303`);
* `Fmt`: Formato dimensional (`Grande` ou `Pequena` via `ZS-201/202`);
* `Cor`: Pigmentacao (`Vermelha`, `Verde`, `Azul` via `AS-201..203`).

In [ ]:
# Dataset auditavel de 16 ciclos da Celula de Manufatura Flexivel
fms_records = [
    {"Seg": "OK",    "Caixa": "Disp",  "Fmt": "Grande",  "Cor": "Vermelha", "Acao_SCADA": "CX-301"},
    {"Seg": "OK",    "Caixa": "Disp",  "Fmt": "Grande",  "Cor": "Vermelha", "Acao_SCADA": "CX-301"},
    {"Seg": "OK",    "Caixa": "Disp",  "Fmt": "Grande",  "Cor": "Verde",    "Acao_SCADA": "CX-302"},
    {"Seg": "OK",    "Caixa": "Disp",  "Fmt": "Grande",  "Cor": "Verde",    "Acao_SCADA": "CX-302"},
    {"Seg": "OK",    "Caixa": "Disp",  "Fmt": "Pequena", "Cor": "Azul",     "Acao_SCADA": "CX-303"},
    {"Seg": "OK",    "Caixa": "Disp",  "Fmt": "Pequena", "Cor": "Azul",     "Acao_SCADA": "CX-303"},
    {"Seg": "OK",    "Caixa": "Disp",  "Fmt": "Pequena", "Cor": "Vermelha", "Acao_SCADA": "DESC-304"},
    {"Seg": "OK",    "Caixa": "Disp",  "Fmt": "Pequena", "Cor": "Verde",    "Acao_SCADA": "DESC-304"},
    {"Seg": "OK",    "Caixa": "Disp",  "Fmt": "Grande",  "Cor": "Azul",     "Acao_SCADA": "DESC-304"},
    {"Seg": "OK",    "Caixa": "Cheia", "Fmt": "Pequena", "Cor": "Vermelha", "Acao_SCADA": "TRIP_ALARME"},
    {"Seg": "OK",    "Caixa": "Cheia", "Fmt": "Pequena", "Cor": "Azul",     "Acao_SCADA": "TRIP_ALARME"},
    {"Seg": "Falha", "Caixa": "Disp",  "Fmt": "Grande",  "Cor": "Vermelha", "Acao_SCADA": "TRIP_ALARME"},
    {"Seg": "Falha", "Caixa": "Disp",  "Fmt": "Grande",  "Cor": "Verde",    "Acao_SCADA": "TRIP_ALARME"},
    {"Seg": "Falha", "Caixa": "Disp",  "Fmt": "Pequena", "Cor": "Azul",     "Acao_SCADA": "TRIP_ALARME"},
    {"Seg": "Falha", "Caixa": "Disp",  "Fmt": "Pequena", "Cor": "Vermelha", "Acao_SCADA": "TRIP_ALARME"},
    {"Seg": "Falha", "Caixa": "Cheia", "Fmt": "Grande",  "Cor": "Azul",     "Acao_SCADA": "TRIP_ALARME"},
]
df_fms = pd.DataFrame(fms_records)
features_case1 = ["Seg", "Caixa", "Fmt", "Cor"]
X_c1 = df_fms[features_case1]
y_c1 = df_fms["Acao_SCADA"]

# 1. Tabela de Metricas no No Raiz
H_root = shannon_entropy(y_c1.tolist())
G_root = gini_impurity(y_c1.tolist())
print(f"No Raiz (N={len(df_fms)}): Entropia H(D) = {H_root:.4f} bits | Impureza Gini(D) = {G_root:.4f}\n")

root_rows = []
for feat in features_case1:
    m = evaluate_categorical_multi_split(X_c1[feat].tolist(), y_c1.tolist())
    cat_b, d_gini, g_split = best_binary_categorical_split(X_c1[feat].tolist(), y_c1.tolist())
    root_rows.append({
        "Atributo": feat,
        "H_cond": round(m["H_cond"], 4),
        "IG (ID3)": round(m["IG"], 4),
        "SplitInfo": round(m["SplitInfo"], 4),
        "GainRatio (C4.5)": round(m["GainRatio"], 4),
        "Corte Binario CART": f"{feat} == '{cat_b}'",
        "Gini_split": round(g_split, 4),
        "Delta_Gini (CART)": round(d_gini, 4),
    })

df_root_metrics = pd.DataFrame(root_rows)
display(df_root_metrics)

# 2. Treinamento comparativo dos 3 algoritmos
models_c1 = {}
summary_c1 = []
for algo in ["id3", "c45", "cart"]:
    clf = ClassicDecisionTreeClassifier(algorithm=algo)
    clf.fit(X_c1, y_c1, categorical_features=set(features_case1))
    models_c1[algo] = clf
    print(f"\n{'='*65}")
    print(f" ARVORE INDUZIDA -- ALGORITMO {algo.upper()}")
    print(f"{'='*65}")
    clf.print_tree()
    summary_c1.append({
        "Algoritmo": algo.upper(),
        "No Raiz Selecionado": clf.root.split_feature,
        "Profundidade": clf.get_depth(),
        "Total de Nos": clf.count_nodes(),
        "Folhas": clf.count_leaves(),
        "Testes Medios/Ciclo": round(clf.average_tests_per_sample(X_c1), 3),
        "Acuracia (%)": round(clf.score(X_c1, y_c1) * 100, 2),
    })

print("\nResumo Estrutural Comparativo (Estudo de Caso 1):")
display(pd.DataFrame(summary_c1))

In [ ]:
# Visualizacao comparativa das metricas na raiz e complexidade das arvores (Caso 1)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), dpi=120)

# Grafico 1: Criterios de selecao no no raiz
x_idx = np.arange(len(df_root_metrics))
width = 0.26
axes[0].bar(x_idx - width, df_root_metrics["IG (ID3)"], width, label="IG (ID3)", color="#f59e0b")
axes[0].bar(x_idx, df_root_metrics["GainRatio (C4.5)"], width, label="Gain Ratio (C4.5)", color="#0284c7")
axes[0].bar(x_idx + width, df_root_metrics["Delta_Gini (CART)"], width, label="Delta Gini (CART)", color="#10b981")
axes[0].set_xticks(x_idx)
axes[0].set_xticklabels(df_root_metrics["Atributo"], fontweight="bold")
axes[0].set_ylabel("Valor da Metrica de Divisao")
axes[0].set_title("Selecao do No Raiz: Vies de Cardinalidade (ID3) vs C4.5 e CART", fontweight="bold")
axes[0].legend()

# Grafico 2: Complexidade estrutural das arvores induzidas
df_sum = pd.DataFrame(summary_c1)
x_alg = np.arange(len(df_sum))
w2 = 0.22
axes[1].bar(x_alg - w2, df_sum["Profundidade"], w2, label="Profundidade Max.", color="#6366f1")
axes[1].bar(x_alg, df_sum["Total de Nos"], w2, label="Total de Nos", color="#0ea5e9")
axes[1].bar(x_alg + w2, df_sum["Testes Medios/Ciclo"], w2, label="Testes Medios/Ciclo", color="#ec4899")
axes[1].set_xticks(x_alg)
axes[1].set_xticklabels(df_sum["Algoritmo"], fontweight="bold")
axes[1].set_ylabel("Contagem / Custo de Avaliacao")
axes[1].set_title("Complexidade Estrutural e Custo de Scan no CLP", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()

---
### 5. Estudo de Caso 2: Telemetria Continua, Poda (*Pruning*) e Supressao Hierarquica de Alarmes

Em operacao industrial continua, falhas primarias desencadeiam **inundacoes de alarmes (*alarm floods*)** na IHM SCADA. Por exemplo, uma queda na rede pneumatica (`Pressao_Ar_bar < 4.3`) impede os pistoes `XV-201..203` de atingirem os fins de curso `FC_p1..3`, gerando simultaneamente 4 alarmes derivados.

Simulamos $N = 360$ ciclos historicos com 5 variaveis de telemetria (`Pressao_Ar_bar`, `Corrente_M101_A`, `Tempo_Inspecao_ms`, `ESD_301`, `LS_101`), ruido gaussiano tipico de transdutores 4--20 mA e falhas sobrepostas, para avaliar:
1. A descoberta automatica de limiares fisicos continuos via **C4.5** e **CART**;
2. O impacto da **pre-poda (`max_depth`)** e da **pos-poda (*Reduced-Error Pruning*)** na mitigacao de *overfitting*;
3. A taxa de reducao de alarmes brutos atraves do diagnostico hierarquico de **Causa Raiz**.

In [ ]:
def generate_fms_telemetry_dataset(seed: int = 42) -> pd.DataFrame:
    """
    Gera 360 ciclos realisticos de telemetria hibrida da FMS com ruido analogico
    e registro de alarmes brutos em cascata vs. causa raiz real.
    """
    rng = np.random.default_rng(seed)
    class_specs = [
        ("NORMAL",          200, 0),  # Operacao nominal (0 alarmes ativos)
        ("BAIXA_PRESSAO",    40, 4),  # Dispara PRESS_LOW + FC_p1_TO + FC_p2_TO + FC_p3_TO
        ("SOBRECARGA_M101",  40, 3),  # Dispara TH_101 + M101_STALL + INSP_TIMEOUT
        ("ATOLAMENTO_INSP",  35, 3),  # Dispara INSP_JAM + ZS_201_STUCK + FEED_HOLD
        ("SILO_VAZIO",       25, 2),  # Dispara LS_101 + XV_101_DRY
        ("E_STOP",           20, 5),  # Dispara ESD_301 + M101_TRIP + XV_OFF + 2 interlocks
    ]

    rows = []
    for root_cause, count, raw_alarms in class_specs:
        for _ in range(count):
            pressao = float(rng.normal(6.0, 0.22))
            corrente = float(rng.normal(2.4, 0.25))
            tempo_insp = float(rng.normal(320.0, 35.0))
            esd = "0"
            ls_101 = "0"

            if root_cause == "BAIXA_PRESSAO":
                pressao = float(rng.normal(3.6, 0.38))
                tempo_insp = float(rng.normal(360.0, 50.0))
            elif root_cause == "SOBRECARGA_M101":
                corrente = float(rng.normal(4.85, 0.42))
                tempo_insp = float(rng.normal(490.0, 75.0))
            elif root_cause == "ATOLAMENTO_INSP":
                tempo_insp = float(rng.normal(730.0, 80.0))
                corrente = float(rng.normal(2.9, 0.35))
            elif root_cause == "SILO_VAZIO":
                ls_101 = "1"
            elif root_cause == "E_STOP":
                esd = "1"
                corrente = float(rng.normal(0.3, 0.15))

            rows.append({
                "Pressao_Ar_bar": round(max(0.5, pressao), 3),
                "Corrente_M101_A": round(max(0.0, corrente), 3),
                "Tempo_Inspecao_ms": round(max(50.0, tempo_insp), 1),
                "ESD_301": esd,
                "LS_101": ls_101,
                "Alarmes_Brutos": raw_alarms,
                "Causa_Raiz": root_cause,
            })

    df = pd.DataFrame(rows)
    return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)


def stratified_split(df: pd.DataFrame, target_col: str, seed: int = 42):
    train_parts, val_parts, test_parts = [], [], []
    for _, grp in df.groupby(target_col, sort=True):
        grp_shuf = grp.sample(frac=1.0, random_state=seed)
        n = len(grp_shuf)
        n_train = int(round(0.6 * n))
        n_val = int(round(0.2 * n))
        train_parts.append(grp_shuf.iloc[:n_train])
        val_parts.append(grp_shuf.iloc[n_train:n_train + n_val])
        test_parts.append(grp_shuf.iloc[n_train + n_val:])
    df_tr = pd.concat(train_parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    df_va = pd.concat(val_parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    df_te = pd.concat(test_parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return df_tr, df_va, df_te


df_tel = generate_fms_telemetry_dataset(seed=42)
df_train, df_val, df_test = stratified_split(df_tel, "Causa_Raiz", seed=42)

feat_c2 = ["Pressao_Ar_bar", "Corrente_M101_A", "Tempo_Inspecao_ms", "ESD_301", "LS_101"]
cat_c2 = {"ESD_301", "LS_101"}

print(f"Dataset de Telemetria FMS gerado: {len(df_tel)} ciclos")
print(f"  -> Treino: {len(df_train)} | Validacao: {len(df_val)} | Teste: {len(df_test)}")
display(df_tel.groupby("Causa_Raiz")[["Pressao_Ar_bar", "Corrente_M101_A", "Tempo_Inspecao_ms", "Alarmes_Brutos"]].mean().round(2))

In [ ]:
# 1. Varredura de profundidade (Pre-poda) para CART e C4.5
sweep_results = []
for depth in range(1, 11):
    for algo in ["cart", "c45"]:
        m = ClassicDecisionTreeClassifier(algorithm=algo, max_depth=depth, min_samples_leaf=2)
        m.fit(df_train[feat_c2], df_train["Causa_Raiz"], categorical_features=cat_c2)
        sweep_results.append({
            "Algoritmo": algo.upper(),
            "max_depth": depth,
            "Nos": m.count_nodes(),
            "Acc_Treino": m.score(df_train[feat_c2], df_train["Causa_Raiz"]),
            "Acc_Validacao": m.score(df_val[feat_c2], df_val["Causa_Raiz"]),
            "Acc_Teste": m.score(df_test[feat_c2], df_test["Causa_Raiz"]),
        })
df_sweep = pd.DataFrame(sweep_results)

# 2. Comparacao: CART Completo (Sem Poda) vs CART Pos-Podado (Reduced-Error Pruning)
cart_full = ClassicDecisionTreeClassifier(algorithm="cart", max_depth=None, min_samples_leaf=1)
cart_full.fit(df_train[feat_c2], df_train["Causa_Raiz"], categorical_features=cat_c2)
acc_full_tr = cart_full.score(df_train[feat_c2], df_train["Causa_Raiz"])
acc_full_te = cart_full.score(df_test[feat_c2], df_test["Causa_Raiz"])
nodes_before = cart_full.count_nodes()
depth_before = cart_full.get_depth()

cart_pruned = ClassicDecisionTreeClassifier(algorithm="cart", max_depth=None, min_samples_leaf=1)
cart_pruned.fit(df_train[feat_c2], df_train["Causa_Raiz"], categorical_features=cat_c2)
collapsed = cart_pruned.prune_reduced_error(df_val[feat_c2], df_val["Causa_Raiz"])
acc_pruned_tr = cart_pruned.score(df_train[feat_c2], df_train["Causa_Raiz"])
acc_pruned_te = cart_pruned.score(df_test[feat_c2], df_test["Causa_Raiz"])
nodes_after = cart_pruned.count_nodes()
depth_after = cart_pruned.get_depth()

print("=" * 72)
print(" IMPACTO DA POS-PODA (REDUCED-ERROR PRUNING) NO CART DA PLANTA FMS")
print("=" * 72)
print(f"CART Sem Poda -> Nos: {nodes_before:2d} | Prof.: {depth_before} | Treino: {acc_full_tr*100:.2f}% | Teste: {acc_full_te*100:.2f}%")
print(f"CART Podado   -> Nos: {nodes_after:2d} | Prof.: {depth_after} | Treino: {acc_pruned_tr*100:.2f}% | Teste: {acc_pruned_te*100:.2f}% ({collapsed} subarvores podadas)")
print("=" * 72)

print("\nArvore CART Pos-Podada (Limiares Fisicos Extraidos Automaticamente):")
cart_pruned.print_tree()

# 3. Auditoria de Supressao de Inundacao de Alarmes no Conjunto de Teste
df_faults_test = df_test[df_test["Causa_Raiz"] != "NORMAL"].copy()
df_faults_test["Predito"] = cart_pruned.predict(df_faults_test[feat_c2])
total_raw_alarms = int(df_faults_test["Alarmes_Brutos"].sum())
suppressed_alarms_count = len(df_faults_test)
reduction_pct = (1.0 - suppressed_alarms_count / total_raw_alarms) * 100.0

print(f"\nRelatorio ISA-18.2 (Mitigacao de Inundacao de Alarmes no Teste):")
print(f"  -> Eventos de falha avaliados:              {len(df_faults_test)}")
print(f"  -> Alarmes brutos em cascata sem arvore:    {total_raw_alarms} alarmes na IHM")
print(f"  -> Alarmes raiz exibidos com arvore CART:   {suppressed_alarms_count} diagnosticos unicos")
print(f"  -> Taxa de supressao de alarmes derivados:  {reduction_pct:.2f}%")

y_test_pred = cart_pruned.predict(df_test[feat_c2])
display(pd.crosstab(df_test["Causa_Raiz"], pd.Series(y_test_pred, name="Predito_CART"), margins=True))

In [ ]:
# Visualizacoes do Estudo de Caso 2: Curva de Poda + Fronteiras de Decisao Ortogonais
fig, axes = plt.subplots(1, 2, figsize=(15, 5.8), dpi=120)

# Subplot 1: Curva de Poda (Treino vs Validacao em funcao de max_depth)
cart_sw = df_sweep[df_sweep["Algoritmo"] == "CART"]
axes[0].plot(cart_sw["max_depth"], cart_sw["Acc_Treino"] * 100, marker="o", lw=2.2, color="#0284c7", label="Acuracia Treino (%)")
axes[0].plot(cart_sw["max_depth"], cart_sw["Acc_Validacao"] * 100, marker="s", lw=2.2, color="#16a34a", label="Acuracia Validacao (%)")
axes[0].axvline(depth_after, color="#e11d48", linestyle="--", lw=1.8, label=f"Profundidade Pos-Podada (d={depth_after})")
axes[0].set_xlabel("Profundidade Maxima (max_depth)")
axes[0].set_ylabel("Acuracia (%)")
axes[0].set_title("Curva de Poda CART: Generalizacao vs Overfitting de Ruido", fontweight="bold")
axes[0].set_xticks(range(1, 11))
axes[0].set_ylim(50, 103)
axes[0].legend(loc="lower right")

# Subplot 2: Espaco 2D Pressao x Corrente mostrando cortes ortogonais aos eixos
palette = {
    "NORMAL": "#10b981",
    "BAIXA_PRESSAO": "#f59e0b",
    "SOBRECARGA_M101": "#ef4444",
    "ATOLAMENTO_INSP": "#8b5cf6",
    "SILO_VAZIO": "#0ea5e9",
    "E_STOP": "#1e293b",
}
for cls, color in palette.items():
    sub = df_tel[df_tel["Causa_Raiz"] == cls]
    axes[1].scatter(
        sub["Pressao_Ar_bar"], sub["Corrente_M101_A"],
        s=32, alpha=0.78, color=color, edgecolors="white", linewidth=0.5, label=cls
    )

def collect_thresholds(node: Optional[DecisionNode], out: List[Tuple[str, float]]) -> None:
    if node is None or node.is_leaf:
        return
    if node.split_type == "bin_num":
        out.append((node.split_feature, node.split_threshold))
    for ch in node.children.values():
        collect_thresholds(ch, out)

thresholds_found: List[Tuple[str, float]] = []
collect_thresholds(cart_pruned.root, thresholds_found)
for feat_name, t_val in thresholds_found:
    if feat_name == "Pressao_Ar_bar":
        axes[1].axvline(t_val, color="#d97706", linestyle="--", lw=1.8, alpha=0.9)
        axes[1].text(t_val + 0.06, 5.6, f"P <= {t_val:.2f} bar", color="#b45309", fontsize=8.5, fontweight="bold")
    elif feat_name == "Corrente_M101_A":
        axes[1].axhline(t_val, color="#dc2626", linestyle="--", lw=1.8, alpha=0.9)
        axes[1].text(1.8, t_val + 0.12, f"I <= {t_val:.2f} A", color="#b91c1c", fontsize=8.5, fontweight="bold")

axes[1].set_xlabel("Pressao Pneumatica da Rede (bar)")
axes[1].set_ylabel("Corrente do Motor M-101 (A)")
axes[1].set_title("Cortes Ortogonais Descobertos pelo CART na Telemetria FMS", fontweight="bold")
axes[1].legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

---
### 6. Geracao Automatica de Codigo CLP (IEC 61131-3) e Verificacao Formal

Como o algoritmo **CART** produz arvores estritamente binarias, cada no interno mapeia diretamente para uma instrucao condicional `IF / ELSE` em **Texto Estruturado (ST)** compativel com CLPs industriais (ex.: Siemens S7-1500). Abaixo implementamos o gerador automatico e executamos a bateria de assercoes deterministicas de validacao.

In [ ]:
def export_to_structured_text(node: DecisionNode, indent: int = 1) -> str:
    """Converte recursivamente uma arvore binaria CART em codigo IEC 61131-3 Structured Text."""
    pad = "    " * indent
    if node.is_leaf:
        return f"{pad}Acao_SCADA := '{node.predicted_class}';\n"

    if node.split_type == "bin_cat":
        cond = f"({node.split_feature} = '{node.split_category}')"
    elif node.split_type == "bin_num":
        cond = f"({node.split_feature} <= {node.split_threshold:.3f})"
    else:
        raise ValueError("Conversao direta IF/ELSE requer arvore binaria CART.")

    st = f"{pad}IF {cond} THEN\n"
    st += export_to_structured_text(node.children[True], indent + 1)
    st += f"{pad}ELSE\n"
    st += export_to_structured_text(node.children[False], indent + 1)
    st += f"{pad}END_IF;\n"
    return st


print("(* Codigo IEC 61131-3 gerado automaticamente a partir do CART - Caso 1 *)")
print("FUNCTION_BLOCK FB_Decisao_FMS")
print("BEGIN")
print(export_to_structured_text(models_c1["cart"].root, indent=1), end="")
print("END_FUNCTION_BLOCK\n")

# ==============================================================================
# BATERIA DE ASSERCOES FORMAIS DE ENGENHARIA (ECAA08)
# ==============================================================================
assert abs(H_root - 2.0996) < 1e-3, f"Entropia raiz divergente: {H_root}"
assert abs(G_root - 0.7266) < 1e-3, f"Gini raiz divergente: {G_root}"

assert models_c1["id3"].root.split_feature == "Cor", "ID3 deve selecionar 'Cor' na raiz devido ao vies de cardinalidade."
assert models_c1["c45"].root.split_feature == "Seg", "C4.5 deve priorizar 'Seg' na raiz via Gain Ratio."
assert models_c1["cart"].root.split_feature == "Seg", "CART deve priorizar 'Seg' na raiz via Delta Gini."

for algo, clf in models_c1.items():
    assert abs(clf.score(X_c1, y_c1) - 1.0) < 1e-9, f"Modelo {algo} falhou na classificacao perfeita do Caso 1."

assert models_c1["cart"].average_tests_per_sample(X_c1) < models_c1["id3"].average_tests_per_sample(X_c1)

assert nodes_after <= nodes_before, "A pos-poda nao deve aumentar o numero de nos."
assert acc_pruned_te >= 0.90, f"Acuracia de teste do CART podado abaixo do esperado: {acc_pruned_te}"
assert reduction_pct >= 60.0, f"Supressao de alarmes abaixo do minimo esperado: {reduction_pct:.2f}%"

print("[OK] Todas as assercoes matematicas, estruturais e de engenharia industrial passaram 100%!")

---
### 7. Analise Tecnica e Conclusoes de Engenharia

1. **Correcao do Vies de Cardinalidade (ID3 vs. C4.5):**
   No Estudo de Caso 1, o **ID3** selecionou o sensor de cor `Cor` ($IG = 0{,}6302$) na raiz por possuir 3 categorias, fragmentando o fluxo e duplicando verificacoes de seguranca `Seg` em cada subarvore. O **C4.5**, ao normalizar pela Entropia de Divisao ($SplitInfo$), corrigiu esse vies e posicionou `Seg` ($GR = 0{,}5786$) e `Caixa` ($GR = 1{,}0000$) no topo da hierarquia, alinhando o modelo induzido a arquitetura de Sistemas Instrumentados de Seguranca (SIS).
2. **Eficiencia do CART em CLPs Industriais:**
   O algoritmo **CART** garantiu topologia estritamente binaria, agrupando automaticamente combinacoes de refugo (ex.: `Cor != 'Azul'` para pecas pequenas $\rightarrow$ `DESC-304`) e reduzindo o custo medio de testes por ciclo de varredura. Sua traducao direta para blocos `IF/ELSE` em Texto Estruturado (IEC 61131-3) assegura execucao deterministica em tempo real.
3. **Robustez a Ruido e Supressao de Alarmes (ISA-18.2):**
   No Estudo de Caso 2, a combinacao de busca de limiares continuos e **pos-poda (*Reduced-Error Pruning*)** isolou automaticamente os limiares fisicos de falha pneumatica, sobrecarga eletrica e atolamento mecanico, reduzindo drasticamente a contagem de nos sem degradar a acuracia no teste e mitigando a inundacao de alarmes secundarios na IHM do **SCADA-Core**.